<a href="https://colab.research.google.com/github/auzaluis/upsa_mod_202601/blob/main/personalidad/01_script_ETL_personalidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tema 01: Carga de datos

## Importando base de datos

In [ ]:
# Google Auth
from google.colab import auth
auth.authenticate_user()

In [ ]:
# API Client
from google.auth import default
creds, _ = default()

In [ ]:
# gspread authorization
import gspread
gc = gspread.authorize(creds)

In [ ]:
# Accediendo al Google Sheet
url = 'https://docs.google.com/spreadsheets/d/1IQ_RxxTSmBKHTExlxboIRNlMov_F6RyqdcOPrflCv_w/edit?usp=sharing'
gsheets = gc.open_by_url(url)
sheets = gsheets.worksheet('Respuestas de formulario 1').get_all_values()

In [ ]:
type(sheets)

In [ ]:
# Convirtiendo a data frame
import pandas as pd
df = pd.DataFrame(sheets[1:], columns=sheets[0])

## Inspección del data frame

In [ ]:
type(df)

In [ ]:
df.head()

In [ ]:
# Analizar la estructura de un df
df.info()

In [ ]:
# Tupla con la cantidad de filas y columnas
df.shape

In [ ]:
# Cantidad de filas
len(df)

In [ ]:
# Cantidad de columnas
len(df.columns)

In [ ]:
df.columns

## Tema 02: Transformación de datos

### Valores perdidos

Identificando NAs

In [ ]:
df[['Escribe tu edad exacta']] # Los NAs están como strings vacíos

In [ ]:
import numpy as np
df.replace('', np.nan, inplace=True)

In [ ]:
df.info()

In [ ]:
(
    df['Escribe tu edad exacta']
    .isna()
    .value_counts()
)

In [ ]:
df['Escribe tu edad exacta'].dtype

In [ ]:
df['Escribe tu edad exacta'] = pd.to_numeric(df['Escribe tu edad exacta'], errors='coerce')

In [ ]:
df['Escribe tu edad exacta'].dtype

Imputación por el promedio

In [ ]:
# Calcular la media
edad_promedio = df['Escribe tu edad exacta'].mean().round(0)
edad_promedio

In [ ]:
# Creando un nuevo df
df2 = df.copy()

In [ ]:
# Reemplazo por la media
df2['edad2'] = df2['Escribe tu edad exacta'].fillna(edad_promedio)

In [ ]:
df2[['Escribe tu edad exacta', 'edad2']]

In [ ]:
df2.info()

In [ ]:
# Relocalizar la columna 'edad2' después de 'Escribe tu edad exacta'
# new_index = list(df2.columns).index('Escribe tu edad exacta') + 1
# df2.insert(new_index, 'edad2', df2.pop('edad2'))

Creando función relocate en python

In [ ]:
def relocate(df, columna, after):
  new_index = list(df.columns).index(after) + 1
  df.insert(new_index, columna, df.pop(columna))
  return df

In [ ]:
relocate(df=df2, columna='edad2', after='Escribe tu edad exacta')

In [ ]:
df2.info()

Eliminando filas que contienen NAs

In [ ]:
df2.shape

In [ ]:
df2.dropna(inplace=True)

In [ ]:
df2.shape

### Valores perdidos

Normalización

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# Instanciando StandardScaler()
normalizador = StandardScaler()

In [ ]:
df2['edad2'].head()

In [ ]:
# normalizando
normalizador.fit_transform(df2[['edad2']])

In [ ]:
df3 = df2.copy()

In [ ]:
df3['edadZ'] = normalizador.fit_transform(df3[['edad2']])

In [ ]:
df3 = relocate(df=df3, columna='edadZ', after='edad2')

Rango (min=0 - max=1)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Instanciando minmax
rango = MinMaxScaler()

In [ ]:
df3['edad_rango'] = rango.fit_transform(df3[['edad2']])

In [ ]:
df3 = relocate(df=df3, columna='edad_rango', after='edadZ')

In [ ]:
df3

### Agrupaciones

Numéricas

In [ ]:
cortes = [-float('inf'), 18, 21, float('inf')]
etiquetas = ['18 o menos', '19 a 21', 'Más de 21']

In [ ]:
df3['edadGR'] = pd.cut(df3['edad2'], bins=cortes, labels=etiquetas)

In [ ]:
df3 = relocate(df=df3, columna='edadGR', after='edad_rango')

In [ ]:
df3.value_counts('edadGR')

In [ ]:
df3[['edad2', 'edadZ', 'edad_rango', 'edadGR']].head()

Categóricas

In [ ]:
df3.info()

In [ ]:
df3.iloc[:,8]

In [ ]:
df3.iloc[:,8].value_counts()

In [ ]:
df3.iloc[:,8].isin(['Totalmente verdadero', 'Un poco verdadero']).sum()

In [ ]:
# Función top2box
def top2box(x):
  if x in ['Totalmente verdadero', 'Un poco verdadero']:
    return 1
  else:
    return 0

In [ ]:
df3.iloc[:,8].apply(top2box).sum()

In [ ]:
# Función lambda
df3.iloc[:,8].apply(
    lambda x: 1 if x in ['Totalmente verdadero', 'Un poco verdadero'] else 0
).sum()

**Bucles**

Manera standard

In [ ]:
df4 = df3.copy()

In [ ]:
# Crear una lista vacía
frases = []

# Bucle para llenar la lista
for col in df4.columns:
  if col.startswith('Según'):
    frases.append(col)

In [ ]:
frases

In [ ]:
for frase in frases:
  df4[frase] = df4[frase].apply(top2box)

In [ ]:
df4[frases].head()

# Tema 03: Manipulación de datos

## Selección de columnas

In [ ]:
df4['Sexo']

In [ ]:
df4[['Sexo', 'edad2']]

In [ ]:
# eliminar columna
df4.drop(columns=['Marca temporal'])

In [ ]:
# comprehensive lists
[col for col in df4.columns if col.startswith('¿Cuánto')]

In [ ]:
df4[[col for col in df4.columns if col.startswith('¿Cuánto')]]

In [ ]:
df4[[col for col in df4.columns if col.endswith('00')]]

In [ ]:
df4.filter(like='edad')

## Selección de filas

In [ ]:
# Seleccionar filas cuando Sexo = 'Mujer'
df4[df4['Sexo'] == 'Mujer']

In [ ]:
# Seleccionar filas cuando Sexo no es = 'Mujer'
df4[df4['Sexo'] != 'Mujer']

In [ ]:
# Seleccionar filas donde edad2 > 20
df4[df4['edad2'] > 20]

In [ ]:
# Seleccionar filas donde edad2 entre 18 y 21
df4[(df4['edad2'] >= 18) & (df4['edad2'] <= 21)]

In [ ]:
# Seleccionar filas donde edad2 entre 18 y 21 y sexo = Mujer
df4[
    (df4['edad2'] >= 18) &
    (df4['edad2'] <= 21) &
    (df4['Sexo'] == 'Mujer')
]

## Renombrado de columnas

In [ ]:
df5 = df4.copy()

In [ ]:
df5.columns

Apps

In [ ]:
apps = ['TikTok', 'Instagram', 'Facebook', 'YouTube']

In [ ]:
[col for col in df5.columns if col.startswith('¿Cuánto')]

In [ ]:
apps_dict = dict(zip(
    [col for col in df5.columns if col.startswith('¿Cuánto')],
    apps
))

In [ ]:
apps_dict

In [ ]:
df5.rename(apps_dict, axis=1, inplace=True)

In [ ]:
df5.columns

Frases

In [86]:
df5.columns = (
    df5.columns
    .str.replace('Según tu forma de ser ¿Cuál de las siguientes frases te describe mejor: [', '')
    .str.replace(']', '')
)

In [87]:
df5.columns

Index(['Marca temporal',
       '¿Estás estudiando en algún colegio, universidad o instituto?',
       'Escribe tu edad exacta', 'edad2', 'edadZ', 'edad_rango', 'edadGR',
       'Sexo', 'No discrimino y trato a todos por igual',
       'Me gusta comprar marcas que representen mi status',
       'Me preocupa mucho el medio ambiente',
       'Estudio mucho, doy mi mayor esfuerzo',
       'Busco el éxito sin importar lo que deba sacrificar',
       'Trato de vestir sencillo para no presumir',
       'Creo que la vida se trata de tomar riesgos',
       'Busco hacer cosas emocionantes para no aburrirme',
       'En mi casa, la familia es muy importante, por eso paso mucho tiempo con ellos',
       'Tener dinero es clave para ser respetado',
       'Me cuesta mucho lidiar con gente que opina estupideces',
       'La tradición y religión ayudan a distinguir lo bueno de lo malo',
       'El dinero va y viene, trato de no cuidarlo mucho, más bien lo disfruto',
       'Hay que respetar a los adu

## Pivotado

Pivot longer

In [88]:
apps

['TikTok', 'Instagram', 'Facebook', 'YouTube']

In [89]:
df6 = df5.melt(
    id_vars = ['Marca temporal', 'Sexo', 'edad2'],
    value_vars = apps,
    var_name = 'app',
    value_name = 'time'
)

In [92]:
df6.head()

,Marca temporal,Sexo,edad2,app,time
0,9/08/2021 12:23:46,Hombre,16.0,TikTok,0:00:00
1,9/08/2021 12:25:01,Mujer,16.0,TikTok,0:50:00
2,9/08/2021 12:26:31,Hombre,17.0,TikTok,8:00:00
3,9/08/2021 12:28:49,Mujer,22.0,TikTok,22:35:00
4,9/08/2021 12:31:43,Hombre,16.0,TikTok,7:00:00


Pivot wider

In [93]:
df7 = df6.pivot(
    index = ['Marca temporal', 'Sexo', 'edad2'],
    columns = 'app',
    values = 'time'
)

In [96]:
df7.head()

,,app,Facebook,Instagram,TikTok,YouTube
Marca temporal,Sexo,edad2,,,,
10/01/2023 18:16:15,Mujer,25.0,0:00:00,6:00:00,0:20:00,3:00:00
10/01/2023 18:16:58,Mujer,21.0,5:00:00,6:00:00,3:00:00,2:00:00
10/01/2023 18:17:15,Hombre,22.0,5:00:00,1:00:00,3:00:00,3:00:00
10/01/2023 18:17:22,Mujer,22.0,1:50:00,6:33:00,7:52:00,0:54:00
10/01/2023 18:17:33,Mujer,25.0,0:37:00,2:15:00,1:28:00,0:00:00


# Tema 04: Outliers

In [97]:
df6.head()

,Marca temporal,Sexo,edad2,app,time
0,9/08/2021 12:23:46,Hombre,16.0,TikTok,0:00:00
1,9/08/2021 12:25:01,Mujer,16.0,TikTok,0:50:00
2,9/08/2021 12:26:31,Hombre,17.0,TikTok,8:00:00
3,9/08/2021 12:28:49,Mujer,22.0,TikTok,22:35:00
4,9/08/2021 12:31:43,Hombre,16.0,TikTok,7:00:00


In [99]:
df6['time'].dtype

dtype('O')

Transformar a número

In [100]:
# split() separar texto
'22:35:00'.split(':')

['22', '35', '00']

In [101]:
def tiempo_a_num(x):
  x = x.split(':')
  return int(x[0]) + int(x[1])/60 + int(x[2])/3600

In [102]:
tiempo_a_num('22:35:00')

22.583333333333332

In [103]:
df6['time'] = df6['time'].apply(tiempo_a_num)

In [104]:
df6.head()

,Marca temporal,Sexo,edad2,app,time
0,9/08/2021 12:23:46,Hombre,16.0,TikTok,0.000000
1,9/08/2021 12:25:01,Mujer,16.0,TikTok,0.833333
2,9/08/2021 12:26:31,Hombre,17.0,TikTok,8.000000
3,9/08/2021 12:28:49,Mujer,22.0,TikTok,22.583333
4,9/08/2021 12:31:43,Hombre,16.0,TikTok,7.000000


## Boxplot

In [105]:
import plotly.express as px

In [106]:
px.box(df6, y='time')